In [1]:
from pathlib import Path

import hist
import matplotlib.pyplot as plt
import mplhep as mh
import numpy as np
import pandas as pd
import uproot

from utils import TREE_NAME, branch_values, hist_bins, safe_name

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INPUT_DIR = PROJECT_ROOT / "output" / "florian"
INPUT_FILE = INPUT_DIR / "ZKK.root"
ALL_BRANCHES_CSV = PROJECT_ROOT / "plots" / "florian" / "all_branches.csv"
OUTPUT_DIR = PROJECT_ROOT / "plots" / "florian" / "duplicate_branch_comparison" / INPUT_FILE.stem

SOURCE_PREFIXES = ("SDST_", "RAWSDST_", "RAWFADANA_")
SOURCE_ORDER = {"SDST": 0, "RAWSDST": 1, "RAWFADANA": 2}

assert INPUT_FILE.exists(), f"Missing input ROOT file: {INPUT_FILE}"
assert ALL_BRANCHES_CSV.exists(), f"Missing branch table: {ALL_BRANCHES_CSV}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Comparing duplicate branches in {INPUT_FILE}")
print(f"Writing comparison plots to {OUTPUT_DIR}")


Comparing duplicate branches in /eos/home-j/joshin/workspace-eos/delphi/delphi-nanoaod/output/florian/ZKK.root
Writing comparison plots to /eos/home-j/joshin/workspace-eos/delphi/delphi-nanoaod/plots/florian/duplicate_branch_comparison/ZKK


In [2]:
def split_source(branch: str) -> tuple[str | None, str]:
    for prefix in SOURCE_PREFIXES:
        if branch.startswith(prefix):
            return prefix.rstrip("_"), branch[len(prefix):]
    return None, branch

all_branches = pd.read_csv(ALL_BRANCHES_CSV)["branch"].astype(str).sort_values().reset_index(drop=True)
branch_rows = []
for branch in all_branches:
    source, variable = split_source(branch)
    if source is not None:
        branch_rows.append({"branch": branch, "source": source, "variable": variable})

source_branches = pd.DataFrame(branch_rows)
duplicate_variables = (
    source_branches.groupby("variable", as_index=False)
    .agg(
        n_sources=("source", "nunique"),
        sources=("source", lambda values: ", ".join(sorted(set(values), key=SOURCE_ORDER.get))),
        branches=("branch", lambda values: ", ".join(sorted(values))),
    )
    .query("n_sources > 1")
    .sort_values(["n_sources", "variable"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"Found {len(duplicate_variables)} duplicated variable names across source prefixes")
display(duplicate_variables)

Found 124 duplicated variable names across source prefixes


,variable,n_sources,sources,branches
0,Vtx_chi2,3,"SDST, RAWSDST, RAWFADANA","RAWFADANA_Vtx_chi2, RAWSDST_Vtx_chi2, SDST_Vtx..."
1,Vtx_nOutgoing,3,"SDST, RAWSDST, RAWFADANA","RAWFADANA_Vtx_nOutgoing, RAWSDST_Vtx_nOutgoing..."
2,Vtx_ndf,3,"SDST, RAWSDST, RAWFADANA","RAWFADANA_Vtx_ndf, RAWSDST_Vtx_ndf, SDST_Vtx_ndf"
3,Vtx_position,3,"SDST, RAWSDST, RAWFADANA","RAWFADANA_Vtx_position, RAWSDST_Vtx_position, ..."
4,nVtx,3,"SDST, RAWSDST, RAWFADANA","RAWFADANA_nVtx, RAWSDST_nVtx, SDST_nVtx"
...,...,...,...,...
119,nStic,2,"RAWSDST, RAWFADANA","RAWFADANA_nStic, RAWSDST_nStic"
120,nTracRaw,2,"RAWSDST, RAWFADANA","RAWFADANA_nTracRaw, RAWSDST_nTracRaw"
121,nTrackElement,2,"RAWSDST, RAWFADANA","RAWFADANA_nTrackElement, RAWSDST_nTrackElement"
122,nVdAssocHit,2,"RAWSDST, RAWFADANA","RAWFADANA_nVdAssocHit, RAWSDST_nVdAssocHit"


In [3]:
tree = uproot.open(INPUT_FILE)[TREE_NAME]
file_branches = set(tree.keys())
file_typenames = tree.typenames()

file_duplicate_rows = source_branches[source_branches["branch"].isin(file_branches)].copy()
file_duplicate_variables = (
    file_duplicate_rows.groupby("variable", as_index=False)
    .agg(
        n_sources=("source", "nunique"),
        sources=("source", lambda values: ", ".join(sorted(set(values), key=SOURCE_ORDER.get))),
        branches=("branch", lambda values: ", ".join(sorted(values))),
    )
    .query("n_sources > 1")
    .sort_values(["n_sources", "variable"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"Found {len(file_duplicate_variables)} duplicated variable names present in {INPUT_FILE.name}")
display(file_duplicate_variables)

Found 124 duplicated variable names present in ZKK.root


,variable,n_sources,sources,branches
0,Vtx_chi2,3,"SDST, RAWSDST, RAWFADANA","RAWFADANA_Vtx_chi2, RAWSDST_Vtx_chi2, SDST_Vtx..."
1,Vtx_nOutgoing,3,"SDST, RAWSDST, RAWFADANA","RAWFADANA_Vtx_nOutgoing, RAWSDST_Vtx_nOutgoing..."
2,Vtx_ndf,3,"SDST, RAWSDST, RAWFADANA","RAWFADANA_Vtx_ndf, RAWSDST_Vtx_ndf, SDST_Vtx_ndf"
3,Vtx_position,3,"SDST, RAWSDST, RAWFADANA","RAWFADANA_Vtx_position, RAWSDST_Vtx_position, ..."
4,nVtx,3,"SDST, RAWSDST, RAWFADANA","RAWFADANA_nVtx, RAWSDST_nVtx, SDST_nVtx"
...,...,...,...,...
119,nStic,2,"RAWSDST, RAWFADANA","RAWFADANA_nStic, RAWSDST_nStic"
120,nTracRaw,2,"RAWSDST, RAWFADANA","RAWFADANA_nTracRaw, RAWSDST_nTracRaw"
121,nTrackElement,2,"RAWSDST, RAWFADANA","RAWFADANA_nTrackElement, RAWSDST_nTrackElement"
122,nVdAssocHit,2,"RAWSDST, RAWFADANA","RAWFADANA_nVdAssocHit, RAWSDST_nVdAssocHit"


In [4]:
def plot_duplicate_variable(
    variable: str,
    entry_stop: int | None = None,
    normalize: bool = True,
    figsize: tuple[int, int] = (12, 9),
    show: bool = True,
) -> dict[str, object]:
    rows = file_duplicate_rows[file_duplicate_rows["variable"] == variable].copy()
    rows = rows.sort_values("source", key=lambda series: series.map(SOURCE_ORDER))
    if len(rows) < 2:
        return {
            "file": INPUT_FILE.name,
            "variable": variable,
            "branches": ", ".join(rows["branch"]),
            "statuses": "missing_duplicate",
            "normalize": normalize,
            "plotted": False,
            "path": "",
        }

    values_by_branch = {}
    status_by_branch = {}
    for row in rows.itertuples(index=False):
        values, status = branch_values(tree, row.branch, file_typenames, entry_stop=entry_stop)
        values_by_branch[row.branch] = values
        status_by_branch[row.branch] = status

    statuses = sorted(set(status_by_branch.values()))
    values_by_branch = {branch: values for branch, values in values_by_branch.items() if values.size}
    if not values_by_branch:
        return {
            "file": INPUT_FILE.name,
            "variable": variable,
            "branches": ", ".join(rows["branch"]),
            "statuses": ", ".join(statuses),
            "normalize": normalize,
            "plotted": False,
            "path": "",
        }

    all_values = np.concatenate(list(values_by_branch.values()))
    nbins, lo, hi = hist_bins(all_values)

    mh.style.use(mh.styles.CMS)
    fig, ax = plt.subplots(figsize=figsize)
    for branch, values in values_by_branch.items():
        source, _ = split_source(branch)
        h = hist.Hist.new.Reg(nbins, lo, hi, name="value", label=variable).Double()
        h.fill(value=values)
        mh.histplot(
            h,
            ax=ax,
            histtype="step",
            label=f"{source} ({len(values)})",
            density=normalize,
            yerr=not normalize,
        )

    plotted_statuses = {status_by_branch[branch] for branch in values_by_branch}
    value_label = variable if plotted_statuses == {"values"} else f"{variable} values / multiplicity"
    ax.set_xlabel(value_label)
    ax.set_ylabel("Normalized entries" if normalize else "Entries")
    ax.legend(title=INPUT_FILE.stem)
    mh.label.exp_label(
        exp="DELPHI",
        text="Private Work",
        rlabel=r"LEP1, $\sqrt{s}\sim 91.25$ GeV",
        loc=0,
        ax=ax,
    )
    fig.tight_layout()

    out_path = OUTPUT_DIR / f"{safe_name(variable)}.png"
    fig.savefig(out_path, dpi=150)
    if show:
        display(fig)
    plt.close(fig)

    return {
        "file": INPUT_FILE.name,
        "variable": variable,
        "branches": ", ".join(values_by_branch),
        "statuses": ", ".join(sorted(plotted_statuses)),
        "normalize": normalize,
        "plotted": True,
        "path": str(out_path),
    }


In [5]:
ENTRY_STOP = None
NORMALIZE = True
MAX_PLOTS = None

variables = file_duplicate_variables["variable"].tolist()
if MAX_PLOTS is not None:
    variables = variables[:MAX_PLOTS]

manifest = pd.DataFrame(
    plot_duplicate_variable(
        variable,
        entry_stop=ENTRY_STOP,
        normalize=NORMALIZE,
        show=False,
    )
    for variable in variables
)
manifest.to_csv(OUTPUT_DIR / "duplicate_branch_comparison_manifest.csv", index=False)

print(f"Wrote {len(manifest)} duplicate branch comparison plots to {OUTPUT_DIR}")
display(manifest)

Wrote 124 duplicate branch comparison plots to /eos/home-j/joshin/workspace-eos/delphi/delphi-nanoaod/plots/florian/duplicate_branch_comparison/ZKK


,file,variable,branches,statuses,normalize,plotted,path
0,ZKK.root,Vtx_chi2,"SDST_Vtx_chi2, RAWSDST_Vtx_chi2, RAWFADANA_Vtx...",values,True,True,/eos/home-j/joshin/workspace-eos/delphi/delphi...
1,ZKK.root,Vtx_nOutgoing,"SDST_Vtx_nOutgoing, RAWSDST_Vtx_nOutgoing, RAW...",values,True,True,/eos/home-j/joshin/workspace-eos/delphi/delphi...
2,ZKK.root,Vtx_ndf,"SDST_Vtx_ndf, RAWSDST_Vtx_ndf, RAWFADANA_Vtx_ndf",values,True,True,/eos/home-j/joshin/workspace-eos/delphi/delphi...
3,ZKK.root,Vtx_position,"SDST_Vtx_position, RAWSDST_Vtx_position, RAWFA...",empty_or_non_numeric,True,False,
4,ZKK.root,nVtx,"SDST_nVtx, RAWSDST_nVtx, RAWFADANA_nVtx",values,True,True,/eos/home-j/joshin/workspace-eos/delphi/delphi...
...,...,...,...,...,...,...,...
119,ZKK.root,nStic,"RAWSDST_nStic, RAWFADANA_nStic",values,True,True,/eos/home-j/joshin/workspace-eos/delphi/delphi...
120,ZKK.root,nTracRaw,"RAWSDST_nTracRaw, RAWFADANA_nTracRaw",values,True,True,/eos/home-j/joshin/workspace-eos/delphi/delphi...
121,ZKK.root,nTrackElement,"RAWSDST_nTrackElement, RAWFADANA_nTrackElement",values,True,True,/eos/home-j/joshin/workspace-eos/delphi/delphi...
122,ZKK.root,nVdAssocHit,"RAWSDST_nVdAssocHit, RAWFADANA_nVdAssocHit",values,True,True,/eos/home-j/joshin/workspace-eos/delphi/delphi...
